# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

## 2. Datos

In [2]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [3]:
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    str    
 2   Product           912 non-null    str    
 3   TypeName          912 non-null    str    
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    str    
 6   Cpu               912 non-null    str    
 7   Ram               912 non-null    str    
 8   Memory            912 non-null    str    
 9   Gpu               912 non-null    str    
 10  OpSys             912 non-null    str    
 11  Weight            912 non-null    str    
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), str(10)
memory usage: 203.7 KB


*Hacemos pequeñas modificaciones para el modelo:*

In [5]:
df['Ram'].unique()

<ArrowStringArray>
['8GB', '16GB', '4GB', '12GB', '32GB', '6GB', '2GB', '64GB', '24GB']
Length: 9, dtype: str

In [6]:
df['RAM'] = pd.to_numeric(df['Ram'].str.replace('GB',''))

In [7]:
df['Weight'].unique()

<ArrowStringArray>
['1.86kg', '2.59kg', '2.04kg', '1.34kg', '2.25kg', '2.31kg', '1.32kg',
  '2.2kg',  '2.7kg',  '2.3kg',
 ...
 '1.41kg', '2.77kg', '1.76kg', '1.55kg', '1.15kg', '0.91kg', '2.15kg',
 '2.54kg', '1.18kg', '4.33kg']
Length: 165, dtype: str

In [8]:
df['WEIGHT'] = pd.to_numeric(df['Weight'].str.replace('kg',''))

In [9]:
df['ScreenResolution'].unique()


<ArrowStringArray>
[                            'Full HD 1920x1080',
                                      '1440x900',
                                      '1366x768',
                   'IPS Panel Full HD 1920x1080',
               'IPS Panel 4K Ultra HD 3840x2160',
            'IPS Panel Retina Display 2560x1600',
                          'Touchscreen 1366x768',
                  'IPS Panel Quad HD+ 2560x1440',
                            'IPS Panel 1366x768',
               'Full HD / Touchscreen 1920x1080',
                   'IPS Panel Full HD 2160x1440',
                            'Quad HD+ 3200x1800',
           '4K Ultra HD / Touchscreen 3840x2160',
            'IPS Panel Retina Display 2304x1440',
     'IPS Panel Full HD / Touchscreen 1920x1080',
               'IPS Panel Touchscreen 1920x1200',
               'IPS Panel Touchscreen 2560x1440',
                         'Touchscreen 2400x1600',
 'IPS Panel 4K Ultra HD / Touchscreen 3840x2160',
                   'IPS Panel F

In [10]:
df[['num1', 'num2']] = df['ScreenResolution'].str.extract(r'(\d+)\s*x\s*(\d+)')
df['num1'] = pd.to_numeric(df['num1'])
df['num2'] = pd.to_numeric(df['num2'])
df['pixels'] = df['num1'] * df['num2']

In [11]:
df['Cpu'].unique()


<ArrowStringArray>
[            'Intel Core i3 6006U 2GHz',
          'Intel Core i7 6700HQ 2.6GHz',
           'Intel Core i7 7500U 2.7GHz',
                 'Intel Core i5 1.8GHz',
           'Intel Core i3 6006U 2.0GHz',
           'Intel Core i5 7300U 2.6GHz',
           'Intel Core i5 7200U 2.5GHz',
           'Intel Core i3 7100U 2.4GHz',
          'Intel Core i7 7700HQ 2.8GHz',
           'Intel Core i7 7600U 2.8GHz',
 ...
                 'Intel Core i7 2.8GHz',
                'AMD Ryzen 1600 3.2GHz',
        'Intel Xeon E3-1535M v5 2.9GHz',
                 'Intel Core i5 2.3GHz',
 'Intel Pentium Dual Core 4405U 2.1GHz',
           'Intel Core i3 6006U 2.2GHz',
             'Intel Atom Z8350 1.92GHz',
          'Intel Core i5 7200U 2.50GHz',
              'AMD A6-Series 7310 2GHz',
 'Intel Pentium Dual Core N4200 1.1GHz']
Length: 107, dtype: str

In [12]:
df['Cpu_brand'] = 'Other'
df.loc[df['Cpu'].str.contains('Intel'), 'Cpu_brand'] = 'Intel'
df.loc[df['Cpu'].str.contains('AMD'), 'Cpu_brand'] = 'AMD'
df.loc[df['Cpu'].str.contains('Apple'), 'Cpu_brand'] = 'Apple'

*Eliminamos columnas:*

In [13]:
df = df.drop(columns=['laptop_ID','Product','num1','num2','ScreenResolution','Weight','Cpu','Ram','Memory'])

*Aplicamos herramienta de ordinal encoding para las siguientes columnas:*

In [14]:
df = pd.get_dummies(df, columns=["Company","TypeName","OpSys","Gpu",'Cpu_brand'], dtype=int)

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Columns: 134 entries, Inches to Cpu_brand_Intel
dtypes: float64(3), int64(131)
memory usage: 954.9 KB


### 2.2 Definir X e y


In [16]:
# Tu código aquí
X = df.drop(columns=['Price_in_euros'])
y = df['Price_in_euros']

### 2.3 Dividir en train y test

In [17]:
# Tu código aquí
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
df_test = pd.read_csv('./data/test.csv')

In [19]:
df_test['Cpu_brand'] = 'Other'
df_test.loc[df_test['Cpu'].str.contains('Intel'), 'Cpu_brand'] = 'Intel'
df_test.loc[df_test['Cpu'].str.contains('AMD'), 'Cpu_brand'] = 'AMD'
df_test.loc[df_test['Cpu'].str.contains('Apple'), 'Cpu_brand'] = 'Apple'

In [20]:
df_test = pd.get_dummies(df_test, columns=["Company","TypeName","OpSys","Gpu",'Cpu_brand'], dtype=int)

## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [21]:
# Encoding aplicado en sección 2.1 con pd.get_dummies antes de definir X e y

## 4. Modelado

### 4.1 Entrenamiento

In [22]:
# Tu código aquí
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [23]:
y_pred = model.predict(X_val)
y_pred

array([1004.70704048, 1210.9778    ,  949.1024    , 1013.0998    ,
        367.4       ,  303.169     ,  903.76261667,  377.3632    ,
       1180.54242143, 1515.8657    ,  484.0898    , 1067.62678571,
        863.12873095, 1463.49766667,  840.337     , 1103.9661    ,
       1587.6044    ,  828.16403333, 3098.51443333, 1574.37883333,
        844.17940377, 1185.74788333, 1649.51616667, 1260.6711    ,
        373.582     , 1867.0056    ,  344.4299    ,  599.8993    ,
       1104.56803333,  722.24195119,  589.5218    ,  538.0904    ,
       2634.5876    , 2136.3829    , 3066.151825  ,  267.1369    ,
       1112.705     ,  579.8059    ,  575.713     ,  315.85655   ,
       1034.09760159, 1650.89773333,  515.997     , 1412.31673333,
        514.21342381,  613.62845   ,  565.85      ,  314.45366667,
       1147.09893333, 2124.41122   , 3227.63746667, 1250.9930346 ,
       1318.73633   ,  682.0804    ,  771.0332    , 2025.084     ,
        411.29933333,  799.77783571, 1560.60213333, 2528.26933

### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [24]:
# Tu código aquí
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

357.09090985826094


### 4.3 Optimización (up to you 🫰🏻)

In [25]:
# Tu código aquí


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [26]:
# Tu código aquí
model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [27]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [28]:
# Tu código aquí

X_pred['RAM'] = pd.to_numeric(X_pred['Ram'].str.replace('GB',''))

X_pred['WEIGHT'] = pd.to_numeric(X_pred['Weight'].str.replace('kg',''))

X_pred[['num1', 'num2']] = X_pred['ScreenResolution'].str.extract(r'(\d+)\s*x\s*(\d+)')
X_pred['num1'] = pd.to_numeric(X_pred['num1'])
X_pred['num2'] = pd.to_numeric(X_pred['num2'])
X_pred['pixels'] = X_pred['num1'] * X_pred['num2']

X_pred['Cpu_brand'] = 'Other'
X_pred.loc[X_pred['Cpu'].str.contains('Intel'), 'Cpu_brand'] = 'Intel'
X_pred.loc[X_pred['Cpu'].str.contains('AMD'), 'Cpu_brand'] = 'AMD'
X_pred.loc[X_pred['Cpu'].str.contains('Apple'), 'Cpu_brand'] = 'Apple'

X_pred = pd.get_dummies(X_pred, columns=["Company","TypeName","OpSys","Gpu",'Cpu_brand'], dtype=int)

X_pred = X_pred.drop(columns=['laptop_ID','Product','num1','num2','ScreenResolution','Weight','Cpu','Ram','Memory'])

X_pred = X_pred.reindex(columns=X.columns, fill_value=0)

*usamos pd.get_dummies, no OneHotEncoder. Por tanto no hay .transform() que aplicar*

## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [29]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [30]:
# Tu código aquí
df_final = pd.DataFrame({
    'laptop_ID': df_test['laptop_ID'],
    'Price_in_euros': model.predict(X_pred)
})
df_final.to_csv('submission.csv', index=False)

### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [31]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [33]:
checker(df_final, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260702_004210.csv'. ¡A Kaggle!
